In [ ]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


# Stage 3 — LoRA fine-tuning (combined global + region VQA)

Trains **LoRA adapters on Qwen 2.5 7B** while keeping frozen:
- CLIP ViT-L/336
- Projection-B (576 global tokens)
- Region extractor / Projection-A (16 tokens × N boxes)

**Data:** `~/reva-data/decontamination/stage3_eval_clean.pkl` (hard-ID + pHash clean)

**Prompt layout** matches `run_combined_inference` in `inference.py` as follows:.

```text
[576 global projected patch tokens]

The image tokens above provide an overview of the picture.

Here is region1 <region1> [16 region tokens] located at [x1, y1, x2, y2] within the picture.
Here is region2 <region2> [16 region tokens] located at [x1, y1, x2, y2] within the picture.
... (one block per GT box, up to 10)

{format_prompt}

Question:
{question}

Answer:
{answer}  <- training loss only on this part
```



Set `MAX_TRAIN_SAMPLES` for a quick smoke run before the full 1.7M-row epoch.

## 0. Install PEFT (once per environment)

In [ ]:
!pip install peft --quiet

## 1. Paths and hyperparameters

In [ ]:
import os

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HOME'] = os.environ.get('REVA_HF_CACHE_ROOT') or os.path.expanduser('~/reva-data/hf_cache')
os.environ['TRANSFORMERS_CACHE'] = os.environ.get('REVA_HF_CACHE_ROOT') or os.path.expanduser('~/reva-data/hf_cache')

from pathlib import Path
import torch
from reva.config import Stage3Config
from reva.models import (
    build_region_feature_extractor,
    load_frozen_vit,
    load_projection_b_weights,
    load_qwen_with_lora,
    load_region_feature_extractor,
)
from reva.stage3_train import (
    Stage3VQADataset,
    build_stage3_training_inputs,
    collate_stage3_batch,
    load_stage3_clean_pool,
    stage3_forward_pass,
)
from reva.training import train_stage3_lora

PROJECT_DIR = Path(os.environ.get("REVA_PROJECT_DIR", str(Path.cwd())))
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path('.').resolve()

os.chdir(PROJECT_DIR)
print('Project dir:', PROJECT_DIR)

CLEAN_PKL = Path(os.environ.get('REVA_STAGE3_CLEAN_PKL') or os.path.expanduser('~/reva-data/decontamination/stage3_eval_clean.pkl'))
PROJECTION_B_WEIGHTS = PROJECT_DIR / 'projection_b_best_weights.pt'
PROJECTION_A_WEIGHTS = PROJECT_DIR / 'projection_a_curriculum_best_weights.pt'

MAX_TRAIN_SAMPLES = None   # e.g. 1000 for smoke test; None = full pool
NUM_EPOCHS = 1
GRAD_ACCUM = 16
PEAK_LR = 1e-4
SAVE_EVERY = 1000

config = Stage3Config(
    clean_pkl_path=CLEAN_PKL,
    projection_b_path=str(PROJECTION_B_WEIGHTS),
    projection_a_weights_path=str(PROJECTION_A_WEIGHTS),
    num_epochs=NUM_EPOCHS,
    gradient_accumulation_steps=GRAD_ACCUM,
    peak_learning_rate=PEAK_LR,
    save_every_n_steps=SAVE_EVERY,
    max_train_samples=MAX_TRAIN_SAMPLES,
)
print('Device:', config.device)

## 2. Load clean pool

In [ ]:
train_samples = load_stage3_clean_pool(config.clean_pkl_path, max_samples=config.max_train_samples)
print('Example sample:')
s = train_samples[1]
print(' source:', s['source'])
print(' Q:', s['question'][:120])
print(' A:', s['answer'][:120])
print(' boxes:', len(s['boxes']))

In [ ]:
# --- VQAv2 GT box consistency check + cap preview (uses _draw_inference_boxes) ---
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

from PIL import Image

from reva.inference import _draw_inference_boxes
from reva.stage3_dataset import load_coco_instance_boxes
from reva.stage3_train import load_stage3_clean_pool, scale_boxes_to_336

# Paths (match stage3_decontamination.ipynb)
DATA_DIR = Path(os.environ.get("REVA_REGION_DATA_ROOT") or os.path.expanduser("~/reva-data/region_data"))
VQAV2_ROOT = Path(os.environ.get("REVA_VQAV2_ROOT") or os.path.expanduser("~/reva-data/vqav2"))
COCO_TRAIN2014 = DATA_DIR / "coco" / "train2014"
COCO_TRAIN_INST = DATA_DIR / "coco" / "annotations" / "instances_train2014.json"
STAGE3_PKL = Path(os.environ.get("REVA_STAGE3_CLEAN_PKL") or os.path.expanduser("~/reva-data/decontamination/stage3_eval_clean.pkl"))

POOL_CAP = 20               # must match stage3_eval_clean.pkl build
PREVIEW_CAP = 20            # explore "what if we kept more COCO instances?"
NUM_AUDIT = 500             # rows to validate (raise for full scan)
NUM_PLOT = 5                # random images to visualize
USE_PREVIEW_BOXES = True    # True -> plot PREVIEW_CAP boxes; False -> plot pickle boxes
SEED = 0              # fixed audit sample (optional)
PLOT_SEED = None      # None = different 5 images every rerun
# PLOT_SEED = 42      # set this when you want reproducible plots

rng = random.Random(SEED)

# --- load pool + live COCO indices ---
pool = load_stage3_clean_pool(STAGE3_PKL)
vqav2_rows = [s for s in pool if s["source"] == "vqav2"]
print(f"VQAv2 rows in pickle: {len(vqav2_rows):,}")

coco_pool = load_coco_instance_boxes(COCO_TRAIN_INST, max_boxes_per_image=POOL_CAP)
coco_preview = load_coco_instance_boxes(COCO_TRAIN_INST, max_boxes_per_image=PREVIEW_CAP)
print(f"COCO indexed images: {len(coco_pool):,} (pool cap={POOL_CAP}, preview cap={PREVIEW_CAP})")

with open(VQAV2_ROOT / "v2_OpenEnded_mscoco_train2014_questions.json") as f:
    vq_questions = {int(q["question_id"]): q for q in json.load(f)["questions"]}

# --- audit (strict: pickle vs POOL_CAP only) ---
audit_rows = vqav2_rows if len(vqav2_rows) <= NUM_AUDIT else rng.sample(vqav2_rows, NUM_AUDIT)
issues = Counter()
by_image = defaultdict(list)
preview_gains = 0

for s in audit_rows:
    qid = int(s["question_id"])
    image_id = int(s["image_id"])
    by_image[image_id].append(s)

    q = vq_questions.get(qid)
    if q is None:
        issues["missing_vqav2_question"] += 1
        continue
    if int(q["image_id"]) != image_id:
        issues["question_image_id_mismatch"] += 1

    img_path = Path(s["image_path"])
    if not img_path.exists():
        issues["missing_image_file"] += 1
        continue

    coco_info = coco_pool.get(image_id)
    if coco_info is None:
        issues["missing_coco_index"] += 1
        continue

    if img_path.name != coco_info["file_name"]:
        issues["filename_mismatch"] += 1

    pil = Image.open(img_path).convert("RGB")
    w, h = pil.size
    if w != coco_info["width"] or h != coco_info["height"]:
        issues["size_mismatch_coco_json"] += 1

    pickle_boxes = s["boxes"]
    live_boxes_pool = coco_info["boxes_px"]
    if pickle_boxes != live_boxes_pool:
        issues["boxes_differ_from_live_coco"] += 1

    if len(coco_preview[image_id]["boxes_px"]) > len(pickle_boxes):
        preview_gains += 1

    for x1, y1, x2, y2 in pickle_boxes:
        if not (0 <= x1 < x2 <= w and 0 <= y1 < y2 <= h):
            issues["box_out_of_bounds"] += 1
            break

for image_id, rows in by_image.items():
    box_lists = {tuple(map(tuple, r["boxes"])) for r in rows}
    if len(box_lists) > 1:
        issues["inconsistent_boxes_same_image"] += 1

n_boxes_pickle = [len(s["boxes"]) for s in vqav2_rows]
n_boxes_preview = [len(coco_preview[int(s["image_id"])]["boxes_px"]) for s in vqav2_rows]

print(f"\n=== Audit (pickle vs cap={POOL_CAP}) ===")
print(f"Sample: {len(audit_rows):,} rows | issues: {dict(issues) or 'none'}")
print(
    f"Pickle box count — min={min(n_boxes_pickle)}, max={max(n_boxes_pickle)}, "
    f"mean={sum(n_boxes_pickle)/len(n_boxes_pickle):.2f}, "
    f"at_cap({POOL_CAP})={sum(1 for n in n_boxes_pickle if n == POOL_CAP):,} "
    f"({100*sum(1 for n in n_boxes_pickle if n == POOL_CAP)/len(n_boxes_pickle):.1f}%)"
)
print(f"Unique VQAv2 images in pool: {len({int(s['image_id']) for s in vqav2_rows}):,}")

print(f"\n=== Preview (cap={PREVIEW_CAP}, not in pickle) ===")
print(
    f"Would have more boxes on "
    f"{sum(1 for s in vqav2_rows if len(coco_preview[int(s['image_id'])]['boxes_px']) > len(s['boxes'])):,} "
    f"rows ({100*sum(1 for s in vqav2_rows if len(coco_preview[int(s['image_id'])]['boxes_px']) > len(s['boxes']))/len(vqav2_rows):.1f}%)"
)
print(
    f"Preview box count — min={min(n_boxes_preview)}, max={max(n_boxes_preview)}, "
    f"mean={sum(n_boxes_preview)/len(n_boxes_preview):.2f}, "
    f"at_cap({PREVIEW_CAP})={sum(1 for n in n_boxes_preview if n == PREVIEW_CAP):,} "
    f"({100*sum(1 for n in n_boxes_preview if n == PREVIEW_CAP)/len(n_boxes_preview):.1f}%)"
)
print(f"In audit sample, preview adds boxes on {preview_gains}/{len(audit_rows)} images")


def plot_vqav2_sample(s, *, use_preview: bool):
    image_id = int(s["image_id"])
    if use_preview:
        boxes = coco_preview[image_id]["boxes_px"]
        cap_label = PREVIEW_CAP
    else:
        boxes = s["boxes"]
        cap_label = POOL_CAP

    pil = Image.open(s["image_path"]).convert("RGB")
    w, h = pil.size
    boxes_336 = scale_boxes_to_336(boxes, w, h)
    labels = [f"r{i+1}" for i in range(len(boxes))]

    n_pool = len(coco_pool[image_id]["boxes_px"])
    n_prev = len(coco_preview[image_id]["boxes_px"])
    print(
        f"\nimage_id={s['image_id']} | {Path(s['image_path']).name} | "
        f"plotting {len(boxes)} boxes (cap={cap_label}) | "
        f"pickle={len(s['boxes'])} pool={n_pool} preview={n_prev}"
    )
    print(f"Q: {s['question']}")
    _draw_inference_boxes(pil, boxes_336, scores=None, labels=labels)


# --- one row per image, then sample 5 random images ---
vqav2_by_image = {}
for s in vqav2_rows:
    image_id = int(s["image_id"])
    if image_id not in vqav2_by_image:
        vqav2_by_image[image_id] = s

all_image_ids = list(vqav2_by_image.keys())
plot_rng = random.Random(PLOT_SEED) if PLOT_SEED is not None else random
plot_image_ids = plot_rng.sample(all_image_ids, min(NUM_PLOT, len(all_image_ids)))

print(f"\n=== Random plot sample ({len(plot_image_ids)} images) ===")
print(f"plot seed: {PLOT_SEED!r} | image_ids: {plot_image_ids}")

for image_id in plot_image_ids:
    plot_vqav2_sample(vqav2_by_image[image_id], use_preview=USE_PREVIEW_BOXES)

## 3. Load models (frozen vision + LoRA Qwen)

In [ ]:
assert PROJECTION_B_WEIGHTS.exists(), PROJECTION_B_WEIGHTS
assert PROJECTION_A_WEIGHTS.exists(), PROJECTION_A_WEIGHTS
assert CLEAN_PKL.exists(), CLEAN_PKL

frozen_vit, clip_image_processor = load_frozen_vit(config)
projection_head_b = load_projection_b_weights(PROJECTION_B_WEIGHTS, config)

region_extractor = build_region_feature_extractor(config)
region_extractor = load_region_feature_extractor(
    region_extractor, str(PROJECTION_A_WEIGHTS), config
)
for p in region_extractor.parameters():
    p.requires_grad = False

qwen, qwen_tokenizer = load_qwen_with_lora(config)
print('Models ready.')

## 4. Smoke test — one forward + backward step

In [ ]:
from torch.utils.data import DataLoader

smoke_ds = Stage3VQADataset(train_samples[:4], clip_image_processor, max_boxes_per_image=config.max_boxes_per_image)
smoke_loader = DataLoader(smoke_ds, batch_size=1, collate_fn=collate_stage3_batch)
smoke_batch = next(iter(smoke_loader))

qwen.train()
loss = stage3_forward_pass(
    smoke_batch,
    frozen_vit=frozen_vit,
    projection_head_b=projection_head_b,
    region_extractor=region_extractor,
    qwen=qwen,
    qwen_tokenizer=qwen_tokenizer,
    config=config,
)
print('Smoke loss:', float(loss.detach().cpu()))
loss.backward()
print('Backward OK — sequence length ~', smoke_batch['boxes_336'][0].shape[0], 'boxes')
qwen.zero_grad(set_to_none=True)

## 5. Train LoRA

In [ ]:
qwen, training_log = train_stage3_lora(
    qwen,
    qwen_tokenizer,
    frozen_vit,
    projection_head_b,
    region_extractor,
    train_samples,
    clip_image_processor,
    config,
)
print('Best adapter:', config.lora_output_dir / 'best_qwen_lora_weights')

## 6. Quick qualitative check (optional)

Loads the `best` adapter and runs combined inference on one training sample with GT boxes.

In [ ]:
from peft import PeftModel
from reva.inference import run_combined_inference
from reva.models import load_frozen_qwen, load_frozen_vit, load_projection_b_weights

frozen_vit, clip_image_processor = load_frozen_vit(config)
projection_head_b = load_projection_b_weights(PROJECTION_B_WEIGHTS, config)

region_extractor = build_region_feature_extractor(config)
region_extractor = load_region_feature_extractor(
    region_extractor, str(PROJECTION_A_WEIGHTS), config
)
for p in region_extractor.parameters():
    p.requires_grad = False

base_qwen, qwen_tokenizer = load_frozen_qwen(config)
eval_qwen = PeftModel.from_pretrained(base_qwen, config.lora_output_dir / 'best_qwen_lora_weights')
eval_qwen.eval()
eval_qwen = eval_qwen.to(config.device)

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from reva.config import EvalConfig
from reva.evaluation import (
    download_vqav2_val,
    download_coco_val2014_instances,
    load_coco_val2014_boxes_for_vqav2,
    _vqav2_image_path
)
from reva.inference import run_combined_inference, vqa_soft_score

def clean_vqa_pred(raw: str) -> str:
    s = raw.strip().split("\n")[0].strip()
    if s.endswith("."):
        s = s[:-1].strip()
    return s

# --- knobs ---
VQAV2_DIR = Path(os.environ.get("REVA_VQAV2_ROOT") or os.path.expanduser("~/reva-data/vqav2"))
COCO_DIR = Path(os.environ.get("REVA_COCO_DIR") or os.path.expanduser("~/reva-data/region_data/coco"))
START, END = 0, 2000        # smoke slice; full val = END None (~214k)
MAX_BOXES = config.max_boxes_per_image
SHOW_BOXES = True           # overlay COCO GT boxes on image

eval_config = EvalConfig()

vqav2_val_samples, val_images_dir = download_vqav2_val(VQAV2_DIR)
coco_instances = download_coco_val2014_instances(COCO_DIR)
box_lookup = load_coco_val2014_boxes_for_vqav2(
    coco_instances,
    max_boxes_per_image=MAX_BOXES,
)

subset = vqav2_val_samples[START:END]
correct = 0.0

for i, sample in enumerate(subset):
    image_id = sample["image_id"]
    img_path = _vqav2_image_path("val", val_images_dir, image_id)
    manual_boxes = box_lookup[image_id]["boxes_px"]
    gt_answers = sample["answers"]
    gt_top = Counter(gt_answers).most_common(1)[0][0]

    pred = run_combined_inference(
        str(img_path),
        sample["question"],
        frozen_vit,
        projection_head_b,
        region_extractor,
        eval_qwen,
        qwen_tokenizer,
        clip_image_processor,
        config,
        manual_boxes=manual_boxes,
        format_prompt=eval_config.vqa_format_prompt,
        verbose=False,
        max_new_tokens=eval_config.max_new_tokens,
    )
    
    pred = clean_vqa_pred(pred["answer"])
    score = vqa_soft_score(pred, gt_answers)
    correct += score

    # print(f"\n--- [val {START + i}] image_id={image_id} n_boxes={len(manual_boxes)} score={score:.2f} ---")
    # print("Q:", sample["question"])
    # print("GT:", gt_top, f"({len(gt_answers)} annotator answers)")
    # print("Pred:", pred["answer"])

    # # --- show image + boxes ---
    # pil = Image.open(img_path).convert("RGB")
    # fig, ax = plt.subplots(1, figsize=(9, 7))
    # ax.imshow(pil)
    # if SHOW_BOXES:
    #     for j, (x1, y1, x2, y2) in enumerate(manual_boxes):
    #         ax.add_patch(
    #             patches.Rectangle(
    #                 (x1, y1), x2 - x1, y2 - y1,
    #                 linewidth=1.5 if j == 0 else 1.0,
    #                 edgecolor="red" if j == 0 else "cyan",
    #                 facecolor="none",
    #                 alpha=1.0 if j == 0 else 0.5,
    #             )
    #         )
    # ax.set_title(
    #     f"Q: {sample['question']}\n"
    #     f"GT: {gt_top}   |   Pred: {pred['answer']}   |   score: {score:.2f}",
    #     fontsize=10,
    #     loc="left",
    # )
    # ax.axis("off")
    # plt.tight_layout()
    # plt.show()

print(f"\nVQAv2 val soft accuracy [{START}:{END}]: {100 * correct / len(subset):.2f}% ({len(subset)} questions)")

In [ ]:
from collections import Counter
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer

from reva.config import EvalConfig
from reva.evaluation import download_vqav2_val, _vqav2_image_path
from reva.inference import run_vqa_inference, vqa_soft_score
from reva.models import load_frozen_vit, load_projection_b_weights

def clean_vqa_pred(raw: str) -> str:
    s = raw.strip().split("\n")[0].strip()
    if s.endswith("."):
        s = s[:-1].strip()
    return s

# --- vision ---
PROJECTION_B_WEIGHTS = Path(os.environ.get("REVA_PROJECTION_B_WEIGHTS", "projection_b_best_weights.pt"))
frozen_vit, clip_image_processor = load_frozen_vit(config)
projection_head_b = load_projection_b_weights(PROJECTION_B_WEIGHTS, config)

# --- base Qwen, NO LoRA (same as global_token_pipeline) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

qwen_tokenizer = AutoTokenizer.from_pretrained(config.language_model_hf_id)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

frozen_qwen = AutoModelForCausalLM.from_pretrained(
    config.language_model_hf_id,
    torch_dtype=config.compute_dtype,
    device_map={"": device},
)
frozen_qwen.eval()

print("Model type:", type(frozen_qwen))
print("First param device:", next(frozen_qwen.parameters()).device)

# --- VQAv2 val slice (match Stage 1 / Stage 3) ---
VQAV2_DIR = Path(os.environ.get("REVA_VQAV2_ROOT") or os.path.expanduser("~/reva-data/vqav2"))
START, END = 0, 1000

eval_config = EvalConfig()
eval_config.device = str(device)
eval_config.compute_dtype = config.compute_dtype

vqav2_val_samples, val_images_dir = download_vqav2_val(VQAV2_DIR)
subset = vqav2_val_samples[START:END]

# --- smoke test ---
s0 = subset[0]
img0 = Image.open(_vqav2_image_path("val", val_images_dir, s0["image_id"])).convert("RGB")
raw0 = run_vqa_inference(
    img0,
    s0["question"],
    eval_config.vqa_format_prompt,
    frozen_vit,
    projection_head_b,
    frozen_qwen,
    clip_image_processor,
    qwen_tokenizer,
    eval_config,
)
print("\nSmoke test:")
print("Q:", s0["question"])
print("GT:", Counter(s0["answers"]).most_common(1)[0][0])
print("Pred (clean):", clean_vqa_pred(raw0))
print("Raw:", raw0[:120])

# --- 1k eval ---
correct = 0.0
for sample in subset:
    img_path = _vqav2_image_path("val", val_images_dir, sample["image_id"])
    pil = Image.open(img_path).convert("RGB")
    raw = run_vqa_inference(
        pil,
        sample["question"],
        eval_config.vqa_format_prompt,
        frozen_vit,
        projection_head_b,
        frozen_qwen,
        clip_image_processor,
        qwen_tokenizer,
        eval_config,
    )
    pred = clean_vqa_pred(raw)
    correct += vqa_soft_score(pred, sample["answers"])

print(
    f"\nGlobal baseline (no LoRA, run_vqa_inference) "
    f"— VQAv2 val soft acc [{START}:{END}]: {100 * correct / len(subset):.2f}%"
)

In [ ]:
from collections import Counter
from pathlib import Path

import torch
from PIL import Image
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

from reva.config import EvalConfig
from reva.evaluation import download_vqav2_val, _vqav2_image_path
from reva.inference import vqa_soft_score
from reva.models import load_frozen_vit, load_projection_b_weights

def clean_vqa_pred(raw: str) -> str:
    s = raw.strip().split("\n")[0].strip()
    if s.endswith("."):
        s = s[:-1].strip()
    return s

@torch.no_grad()
def run_vqa_inference_lora(
    pil_image,
    question_text: str,
    format_prompt: str,
    *,
    frozen_vit,
    projection_head_b,
    qwen,
    clip_image_processor,
    qwen_tokenizer,
    eval_config,
) -> str:
    """Same as run_vqa_inference, but qwen can be PeftModel (Stage 1/3 LoRA)."""
    frozen_vit.eval()
    projection_head_b.eval()
    qwen.eval()

    processed_image = clip_image_processor(images=pil_image, return_tensors="pt")["pixel_values"]
    processed_image = processed_image.to(eval_config.device, dtype=eval_config.compute_dtype)

    vit_output = frozen_vit(pixel_values=processed_image)
    projected_image_tokens = projection_head_b(vit_output.last_hidden_state[:, 1:, :])

    messages = [{"role": "user", "content": f"{question_text} {format_prompt}"}]
    formatted_prompt = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    tokenised = qwen_tokenizer(formatted_prompt, return_tensors="pt", add_special_tokens=False)
    question_embeddings = qwen.get_input_embeddings()(tokenised["input_ids"].to(eval_config.device))

    input_embeds = torch.cat([projected_image_tokens, question_embeddings], dim=1)

    generated_ids = qwen.generate(
        inputs_embeds=input_embeds,
        max_new_tokens=eval_config.max_new_tokens,
        do_sample=eval_config.do_sample,
        pad_token_id=qwen_tokenizer.pad_token_id,
        eos_token_id=qwen_tokenizer.eos_token_id,
    )
    return qwen_tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()

def eval_chat_lora(adapter_path, label, *, subset, val_images_dir, eval_config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    qwen_tokenizer = AutoTokenizer.from_pretrained(config.language_model_hf_id)
    if qwen_tokenizer.pad_token is None:
        qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

    base_qwen = AutoModelForCausalLM.from_pretrained(
        config.language_model_hf_id,
        torch_dtype=config.compute_dtype,
        device_map={"": device},
    )
    qwen = PeftModel.from_pretrained(base_qwen, adapter_path)
    qwen.eval()

    correct = 0.0
    for sample in tqdm(subset, desc=label):
        img_path = _vqav2_image_path("val", val_images_dir, sample["image_id"])
        pil = Image.open(img_path).convert("RGB")
        raw = run_vqa_inference_lora(
            pil,
            sample["question"],
            eval_config.vqa_format_prompt,
            frozen_vit=frozen_vit,
            projection_head_b=projection_head_b,
            qwen=qwen,
            clip_image_processor=clip_image_processor,
            qwen_tokenizer=qwen_tokenizer,
            eval_config=eval_config,
        )
        pred = clean_vqa_pred(raw)
        correct += vqa_soft_score(pred, sample["answers"])

    acc = 100 * correct / len(subset)
    print(f"{label} — chat global, cleaned [{START}:{END}]: {acc:.2f}%")
    return acc

# --- load vision once ---
PROJECTION_B_WEIGHTS = Path(os.environ.get("REVA_PROJECTION_B_WEIGHTS", "projection_b_best_weights.pt"))
frozen_vit, clip_image_processor = load_frozen_vit(config)
projection_head_b = load_projection_b_weights(PROJECTION_B_WEIGHTS, config)

VQAV2_DIR = Path(os.environ.get("REVA_VQAV2_ROOT") or os.path.expanduser("~/reva-data/vqav2"))
START, END = 0, 3000

eval_config = EvalConfig()
eval_config.device = "cuda" if torch.cuda.is_available() else "cpu"
eval_config.compute_dtype = config.compute_dtype

vqav2_val_samples, val_images_dir = download_vqav2_val(VQAV2_DIR)
subset = vqav2_val_samples[START:END]

STAGE1_ADAPTER = Path(os.environ.get("REVA_STAGE1_ADAPTER") or os.path.expanduser("~/reva-data/checkpoints/stage1_global_lora/best_qwen_lora_weights"))
STAGE3_ADAPTER = Path(os.environ.get("REVA_STAGE3_ADAPTER") or os.path.expanduser("~/reva-data/checkpoints/stage3_lora/best_qwen_lora_weights"))

In [ ]:
acc_s1_chat = eval_chat_lora(
    STAGE1_ADAPTER,
    "Stage 1 LoRA",
    subset=subset,
    val_images_dir=val_images_dir,
    eval_config=eval_config,
)

acc_s3_chat = eval_chat_lora(
    STAGE3_ADAPTER,
    "Stage 3 LoRA",
    subset=subset,
    val_images_dir=val_images_dir,
    eval_config=eval_config,
)

print("\n--- Summary (n=1000, chat + global only, cleaned) ---")
print(f"  Baseline no LoRA (you ran):     39.94%")
print(f"  Stage 1 LoRA:                   {acc_s1_chat:.2f}%")
print(f"  Stage 3 LoRA:                   {acc_s3_chat:.2f}%")
print(f"  Stage 1 custom prefix (prior):  36.98%")
print(f"  Stage 3 combined+boxes (prior): 66.95%")

In [ ]:
from collections import Counter
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

from reva.config import EvalConfig
from reva.evaluation import (
    download_vqav2_val,
    download_coco_val2014_instances,
    load_coco_val2014_boxes_for_vqav2,
    _vqav2_image_path,
)
from reva.inference import (
    _load_pil_image,
    _embed_text,
    _box336_to_normalized,
    manual_boxes_to_336,
    vqa_soft_score,
)
from reva.models import load_frozen_vit, load_projection_b_weights, load_region_feature_extractor, build_region_feature_extractor

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def clean_vqa_pred(raw: str) -> str:
    s = raw.strip().split("\n")[0].strip()
    if s.endswith("."):
        s = s[:-1].strip()
    return s


@torch.no_grad()
def run_combined_chat_inference(
    image_path_or_url: str,
    question: str,
    manual_boxes: list,
    *,
    frozen_vit,
    projection_head_b,
    region_extractor,
    qwen,
    qwen_tokenizer,
    clip_image_processor,
    config,
    format_prompt: str = "Answer the question using a single word or phrase.",
    max_new_tokens: int = 16,
    max_boxes: int = None,
) -> str:
    """
    Combined visual prefix (same as training / run_combined_inference):
      global tokens + overview + region tokens + coordinate text
    Text footer: Qwen chat template (user message = question + format_prompt)
    """
    device = config.device
    max_boxes = max_boxes or getattr(config, "max_boxes_per_image", 20)

    pil_image = _load_pil_image(image_path_or_url)
    img_w, img_h = pil_image.size

    boxes = list(manual_boxes or [])[:max_boxes]
    boxes_336 = manual_boxes_to_336(boxes, img_w, img_h, device)

    pixel_values = clip_image_processor(images=pil_image, return_tensors="pt")["pixel_values"]
    pixel_values = pixel_values.to(device, dtype=config.compute_dtype)

    frozen_vit.eval()
    projection_head_b.eval()
    region_extractor.eval()
    qwen.eval()

    vit_out = frozen_vit(pixel_values=pixel_values, output_hidden_states=True)
    global_tokens = projection_head_b(vit_out.last_hidden_state[:, 1:, :])

    assembled_embeds = []
    assembled_attn = []

    # --- same as run_combined_inference ---
    assembled_embeds.append(global_tokens)
    assembled_attn.append(
        torch.ones(1, global_tokens.shape[1], dtype=torch.long, device=device)
    )

    overview_embeds, overview_mask = _embed_text(
        "\nThe image tokens above provide an overview of the picture.\n",
        qwen_tokenizer, qwen, device,
    )
    assembled_embeds.append(overview_embeds)
    assembled_attn.append(overview_mask)

    for i in range(boxes_336.shape[0]):
        region_num = i + 1
        norm_box = _box336_to_normalized(boxes_336[i].tolist())
        coords = ", ".join(f"{v:.3f}" for v in norm_box)

        prefix_embeds, prefix_mask = _embed_text(
            f"Here is region{region_num} <region{region_num}> ",
            qwen_tokenizer, qwen, device,
        )
        region_tokens = region_extractor(vit_out.hidden_states, boxes_336[i : i + 1])
        suffix_embeds, suffix_mask = _embed_text(
            f" located at [{coords}] within the picture.\n",
            qwen_tokenizer, qwen, device,
        )

        assembled_embeds.extend([prefix_embeds, region_tokens, suffix_embeds])
        assembled_attn.extend([
            prefix_mask,
            torch.ones(1, region_tokens.shape[1], dtype=torch.long, device=device),
            suffix_mask,
        ])

    # --- chat footer (replaces custom "Question:\n...\nAnswer:\n") ---
    messages = [{"role": "user", "content": f"{question} {format_prompt}"}]
    chat_text = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    chat_embeds, chat_mask = _embed_text(chat_text, qwen_tokenizer, qwen, device)
    assembled_embeds.append(chat_embeds)
    assembled_attn.append(chat_mask)

    input_embeds = torch.cat(assembled_embeds, dim=1)
    full_attn = torch.cat(assembled_attn, dim=1)

    output_ids = qwen.generate(
        inputs_embeds=input_embeds,
        attention_mask=full_attn,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        pad_token_id=qwen_tokenizer.pad_token_id,
        eos_token_id=qwen_tokenizer.eos_token_id,  # same as your 68.39% chat-global run
    )
    return qwen_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


# ---------------------------------------------------------------------------
# Load models (Stage 3 LoRA + vision + region extractor)
# ---------------------------------------------------------------------------

PROJECTION_B_WEIGHTS = Path(os.environ.get("REVA_PROJECTION_B_WEIGHTS", "projection_b_best_weights.pt"))
PROJECTION_A_WEIGHTS = Path(os.environ.get("REVA_PROJECTION_A_WEIGHTS", "projection_a_curriculum_best_weights.pt"))
STAGE3_ADAPTER = Path(os.environ.get("REVA_STAGE3_ADAPTER") or os.path.expanduser("~/reva-data/checkpoints/stage3_lora/best_qwen_lora_weights"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

frozen_vit, clip_image_processor = load_frozen_vit(config)
projection_head_b = load_projection_b_weights(PROJECTION_B_WEIGHTS, config)

region_extractor = build_region_feature_extractor(config)
region_extractor = load_region_feature_extractor(region_extractor, str(PROJECTION_A_WEIGHTS), config)

qwen_tokenizer = AutoTokenizer.from_pretrained(config.language_model_hf_id)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

base_qwen = AutoModelForCausalLM.from_pretrained(
    config.language_model_hf_id,
    torch_dtype=config.compute_dtype,
    device_map={"": device},
)
eval_qwen = PeftModel.from_pretrained(base_qwen, STAGE3_ADAPTER)
eval_qwen.eval()

print("Adapter:", STAGE3_ADAPTER)
print("Model:", type(eval_qwen))
print("Device:", next(eval_qwen.parameters()).device)

# ---------------------------------------------------------------------------
# VQAv2 val [0:1000] — combined visual prefix + chat footer
# ---------------------------------------------------------------------------

VQAV2_DIR = Path(os.environ.get("REVA_VQAV2_ROOT") or os.path.expanduser("~/reva-data/vqav2"))
COCO_DIR = Path(os.environ.get("REVA_COCO_DIR") or os.path.expanduser("~/reva-data/region_data/coco"))
START, END = 0, 3000
MAX_BOXES = config.max_boxes_per_image

eval_config = EvalConfig()
eval_config.device = str(device)
eval_config.compute_dtype = config.compute_dtype

vqav2_val_samples, val_images_dir = download_vqav2_val(VQAV2_DIR)
coco_instances = download_coco_val2014_instances(COCO_DIR)
box_lookup = load_coco_val2014_boxes_for_vqav2(
    coco_instances, max_boxes_per_image=MAX_BOXES,
)
subset = vqav2_val_samples[START:END]

# Smoke test
s0 = subset[0]
img0 = _vqav2_image_path("val", val_images_dir, s0["image_id"])
raw0 = run_combined_chat_inference(
    str(img0),
    s0["question"],
    box_lookup[s0["image_id"]]["boxes_px"],
    frozen_vit=frozen_vit,
    projection_head_b=projection_head_b,
    region_extractor=region_extractor,
    qwen=eval_qwen,
    qwen_tokenizer=qwen_tokenizer,
    clip_image_processor=clip_image_processor,
    config=config,
    format_prompt=eval_config.vqa_format_prompt,
    max_new_tokens=eval_config.max_new_tokens,
    max_boxes=MAX_BOXES,
)
print("\nSmoke test:")
print("Q:", s0["question"])
print("GT:", Counter(s0["answers"]).most_common(1)[0][0])
print("Pred (clean):", clean_vqa_pred(raw0))
print("Raw:", raw0[:200])

# Full 1k
correct = 0.0
for sample in tqdm(subset, desc="Stage3 chat+combined visual"):
    image_id = sample["image_id"]
    img_path = _vqav2_image_path("val", val_images_dir, image_id)
    manual_boxes = box_lookup[image_id]["boxes_px"]

    raw = run_combined_chat_inference(
        str(img_path),
        sample["question"],
        manual_boxes,
        frozen_vit=frozen_vit,
        projection_head_b=projection_head_b,
        region_extractor=region_extractor,
        qwen=eval_qwen,
        qwen_tokenizer=qwen_tokenizer,
        clip_image_processor=clip_image_processor,
        config=config,
        format_prompt=eval_config.vqa_format_prompt,
        max_new_tokens=eval_config.max_new_tokens,
        max_boxes=MAX_BOXES,
    )
    pred = clean_vqa_pred(raw)
    correct += vqa_soft_score(pred, sample["answers"])

acc = 100 * correct / len(subset)
print(f"\nStage 3 LoRA — chat footer + combined visual (regions+coords)")
print(f"VQAv2 val soft acc [{START}:{END}]: {acc:.2f}%")

print("\n--- Compare to your prior runs (n=3000, cleaned) ---")
print("  Custom combined + boxes:           66.95%")
print("  Chat + global only (no regions): 68.39%")
print(f"  Chat + combined visual (this run): {acc:.2f}%")